In [0]:
%pip install databricks-vectorsearch --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import time
from databricks.vector_search.client import VectorSearchClient

dbutils.widgets.text("catalog", "rag_chatbot_dev")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("source_table", "pdf_chunks")
dbutils.widgets.text("vs_endpoint", "rag_vs_endpoint")
dbutils.widgets.text("index_name", "pdf_chunks_index")
dbutils.widgets.text("embedding_model", "databricks-gte-large-en")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('source_table')}"
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
INDEX_NAME = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('index_name')}"
EMBEDDING_MODEL = dbutils.widgets.get("embedding_model")

In [0]:
spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:

client = VectorSearchClient(disable_notice=True)

# COMMAND ----------

def get_or_create_endpoint(client, endpoint_name: str):
    """Create the Vector Search endpoint if needed and wait until it is ONLINE."""
    try:
        client.get_endpoint(endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists")
    except Exception:
        print(f"Creating endpoint '{endpoint_name}'...")
        client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

    while True:
        state = client.get_endpoint(endpoint_name)["endpoint_status"]["state"]
        print(f"  endpoint state: {state}")
        if state == "ONLINE":
            return
        if state in ("OFFLINE", "FAILED"):
            raise RuntimeError(f"Endpoint '{endpoint_name}' is in state {state}")
        time.sleep(30)


get_or_create_endpoint(client, VS_ENDPOINT)

# COMMAND ----------

def get_or_create_index(client, endpoint_name: str, index_name: str, source_table: str, model: str):
    """Create a Delta Sync index if it doesn't exist, otherwise return the existing one."""
    try:
        index = client.get_index(endpoint_name=endpoint_name, index_name=index_name)
        print(f"Index '{index_name}' already exists")
        return index
    except Exception:
        print(f"Creating index '{index_name}'...")
        return client.create_delta_sync_index(
            endpoint_name=endpoint_name,
            index_name=index_name,
            source_table_name=source_table,
            pipeline_type="TRIGGERED",          # sync on demand; use "CONTINUOUS" for near real-time
            primary_key="chunk_id",
            embedding_source_column="chunk_text",
            embedding_model_endpoint_name=model,
            columns_to_sync=[
                "chunk_id", "file_name", "file_path", "page_number",
                "chunk_index", "chunk_text", "char_count", "ingested_at",
            ],
        )


index = get_or_create_index(client, VS_ENDPOINT, INDEX_NAME, SOURCE_TABLE, EMBEDDING_MODEL)

# COMMAND ----------

def wait_for_index_ready(index, timeout_s: int = 1800):
    """Poll until the index reports ready (initial sync finished)."""
    waited = 0
    while waited < timeout_s:
        status = index.describe().get("status", {})
        if status.get("ready"):
            print(f"Index ready. {status.get('indexed_row_count', '?')} rows indexed.")
            return
        print(f"  index state: {status.get('detailed_state', 'UNKNOWN')}")
        time.sleep(30)
        waited += 30
    raise TimeoutError("Index did not become ready in time")


wait_for_index_ready(index)

# COMMAND ----------

# Trigger a sync manually (only needed with TRIGGERED pipelines after new PDFs are ingested)
index.sync()
print("Sync triggered")

# COMMAND ----------

# Quick test query
results = index.similarity_search(
    query_text="How many annual leave days do employees get?",
    columns=["file_name", "page_number", "chunk_text"],
    num_results=3,
)

for row in results["result"]["data_array"]:
    file_name, page_number, chunk_text, score = row
    print(f"[{file_name} p.{page_number}] score={score:.3f}")
    print(chunk_text[:300], "...\n")